# 🏭 Sistem Prediksi Risiko Kecelakaan Kerja
## PT Besli Manufacturing

**Tujuan:** Membangun model machine learning untuk memprediksi apakah seorang karyawan berisiko mengalami kecelakaan kerja berdasarkan data historis.

**Dataset:** `dataset_kecelakaan.csv` (300 baris, 9 kolom)

| Fitur | Tipe | Keterangan |
|---|---|---|
| `Usia` | Numerik | Usia karyawan (tahun) |
| `Jam_Kerja_per_Minggu` | Numerik | Total jam kerja per minggu |
| `Pengalaman_Tahun` | Numerik | Lama pengalaman kerja |
| `Pelatihan_K3` | Kategorik | Ya / Tidak |
| `Shift` | Kategorik | Pagi / Siang / Malam |
| `Jabatan` | Kategorik | Operator / Supervisor / Manager |
| `Lokasi` | Kategorik | Gudang / Produksi / Quality_Control |
| `Kecelakaan` | Target (0/1) | 0 = Aman, 1 = Pernah kecelakaan |


## 1. Install & Import Library

In [ ]:
# Install library yang dibutuhkan (jalankan sekali)
# !pip install scikit-learn pandas numpy matplotlib seaborn xgboost joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib

warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

try:
    from xgboost import XGBClassifier
    XGBOOST = True
    print("✅ XGBoost tersedia")
except ImportError:
    XGBOOST = False
    print("⚠️  XGBoost tidak terinstall — akan dilewati")

print("✅ Semua library berhasil diimport")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Load & Eksplorasi Data

In [ ]:
DATA_PATH = "dataset_kecelakaan.csv"   # sesuaikan path jika perlu

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()


In [ ]:
# Ringkasan statistik
df.describe()


In [ ]:
# Cek missing values & tipe data
print("=== Info Dataset ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())


In [ ]:
# Distribusi target
vc = df["Kecelakaan"].value_counts()
print(f"Aman (0)       : {vc[0]} ({vc[0]/len(df)*100:.1f}%)")
print(f"Kecelakaan (1) : {vc[1]} ({vc[1]/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Aman (0)", "Kecelakaan (1)"], [vc[0], vc[1]], color=["#639922", "#E24B4A"], width=0.5)
ax.set_title("Distribusi Target: Kecelakaan Kerja", fontweight="bold")
ax.set_ylabel("Jumlah Karyawan")
for i, v in enumerate([vc[0], vc[1]]):
    ax.text(i, v + 2, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi fitur numerik berdasarkan target
numeric_cols = ["Usia", "Jam_Kerja_per_Minggu", "Pengalaman_Tahun"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    for label, color in [(0, "#639922"), (1, "#E24B4A")]:
        subset = df[df["Kecelakaan"] == label][col]
        ax.hist(subset, alpha=0.6, color=color, bins=20,
                label="Aman" if label == 0 else "Kecelakaan")
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Frekuensi")
    ax.legend()
plt.suptitle("Distribusi Fitur Numerik per Kelas", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Accident rate per variabel kategorik
categoric_cols = ["Pelatihan_K3", "Shift", "Jabatan", "Lokasi"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, categoric_cols):
    rates = df.groupby(col)["Kecelakaan"].mean().sort_values(ascending=False)
    colors = ["#E24B4A" if r > 0.30 else "#EF9F27" if r > 0.25 else "#639922"
              for r in rates]
    bars = ax.bar(rates.index, rates.values * 100, color=colors)
    ax.set_title(col, fontweight="bold")
    ax.set_ylabel("Accident Rate (%)")
    ax.set_ylim(0, 50)
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val*100:.1f}%", ha="center", fontsize=9)
plt.suptitle("Accident Rate per Variabel Kategorik", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Heatmap korelasi fitur numerik
fig, ax = plt.subplots(figsize=(6, 4))
corr = df[numeric_cols + ["Kecelakaan"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            ax=ax, linewidths=0.5)
ax.set_title("Korelasi Fitur Numerik vs Target", fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Preprocessing

In [ ]:
NUMERIC_COLS   = ["Usia", "Jam_Kerja_per_Minggu", "Pengalaman_Tahun"]
CATEGORIC_COLS = ["Pelatihan_K3", "Shift", "Jabatan", "Lokasi"]
TARGET_COL     = "Kecelakaan"
RANDOM_STATE   = 42

X = df.drop(columns=["ID", TARGET_COL])
y = df[TARGET_COL]

# Preprocessor: StandardScaler untuk numerik, OneHotEncoder untuk kategorik
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIC_COLS),
])

# Split data 80:20, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train : {X_train.shape[0]} baris")
print(f"Test  : {X_test.shape[0]} baris")
print(f"Class balance train — Aman: {(y_train==0).sum()}, Kecelakaan: {(y_train==1).sum()}")


## 5. Definisi Model

In [ ]:
models = {}

models["Logistic Regression"] = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
])

models["Random Forest"] = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_split=5,
        class_weight="balanced", random_state=RANDOM_STATE))
])

models["Gradient Boosting"] = Pipeline([
    ("prep", preprocessor),
    ("clf", GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        subsample=0.8, random_state=RANDOM_STATE))
])

if XGBOOST:
    models["XGBoost"] = Pipeline([
        ("prep", preprocessor),
        ("clf", XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            scale_pos_weight=216/84, eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0))
    ])

print(f"Total model yang akan dilatih: {len(models)}")
for name in models:
    print(f"  • {name}")


## 6. Training & Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, pipeline in models.items():
    print(f"\nTraining: {name} ...")
    
    # Cross-validation
    cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_f1  = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    
    # Fit pada full train set
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    results[name] = {
        "pipeline" : pipeline,
        "cv_auc"   : cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "cv_f1"    : cv_f1.mean(),
        "test_auc" : roc_auc_score(y_test, y_prob),
        "y_pred"   : y_pred,
        "y_prob"   : y_prob,
        "cm"       : confusion_matrix(y_test, y_pred),
    }
    
    print(f"  CV AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
    print(f"  Test AUC: {results[name]['test_auc']:.4f}")

print("\n✅ Semua model selesai dilatih")


## 7. Evaluasi & Perbandingan Model

In [ ]:
# Tabel perbandingan
summary = pd.DataFrame({
    name: {
        "CV AUC"     : f"{r['cv_auc']:.4f} ± {r['cv_auc_std']:.4f}",
        "Test AUC"   : f"{r['test_auc']:.4f}",
        "CV F1"      : f"{r['cv_f1']:.4f}",
    }
    for name, r in results.items()
}).T

print("=== Perbandingan Model ===")
display(summary)

best_name = max(results, key=lambda k: results[k]["cv_auc"])
print(f"\n★  Model terbaik: {best_name}  (CV AUC = {results[best_name]['cv_auc']:.4f})")


In [ ]:
# ROC Curves semua model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#3266ad", "#E24B4A", "#639922", "#EF9F27"]
ax = axes[0]
for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
    ax.plot(fpr, tpr, label=f"{name} (AUC={r['test_auc']:.3f})", color=color, lw=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Semua Model", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# CV AUC bar chart
ax = axes[1]
names  = list(results.keys())
cv_auc = [results[n]["cv_auc"] for n in names]
bars   = ax.barh(names, cv_auc, color=colors[:len(names)], alpha=0.85)
ax.axvline(0.5, color="gray", linestyle="--", label="Baseline (0.5)")
ax.set_xlabel("Cross-Validation AUC")
ax.set_title("Perbandingan CV AUC", fontweight="bold")
ax.set_xlim(0, 0.8)
for bar, val in zip(bars, cv_auc):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix semua model
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    disp = ConfusionMatrixDisplay(r["cm"], display_labels=["Aman", "Celaka"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontweight="bold", fontsize=11)

plt.suptitle("Confusion Matrix — Semua Model", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Laporan klasifikasi model terbaik
print(f"=== Classification Report: {best_name} ===\n")
print(classification_report(
    y_test, results[best_name]["y_pred"],
    target_names=["Aman (0)", "Celaka (1)"]
))


## 8. Feature Importance

In [ ]:
clf  = results[best_name]["pipeline"].named_steps["clf"]
prep = results[best_name]["pipeline"].named_steps["prep"]

if hasattr(clf, "feature_importances_"):
    num_names = NUMERIC_COLS
    cat_names = prep.named_transformers_["cat"].get_feature_names_out(CATEGORIC_COLS).tolist()
    feature_names = num_names + cat_names

    fi = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Per one-hot feature
    ax = axes[0]
    colors_fi = ["#E24B4A" if v > 0.15 else "#EF9F27" if v > 0.05 else "#3266ad" for v in fi]
    fi.plot(kind="barh", ax=ax, color=colors_fi)
    ax.set_title(f"Feature Importance (detail)
{best_name}", fontweight="bold")
    ax.set_xlabel("Importance")
    for i, v in enumerate(fi):
        ax.text(v + 0.001, i, f"{v*100:.1f}%", va="center", fontsize=8)

    # Per variabel asli (dikelompokkan)
    ax = axes[1]
    var_imp = {}
    for feat, imp in fi.items():
        matched = False
        for col in CATEGORIC_COLS:
            if feat.startswith(col + "_"):
                var_imp[col] = var_imp.get(col, 0) + imp
                matched = True
                break
        if not matched:
            var_imp[feat] = var_imp.get(feat, 0) + imp

    fi_agg = pd.Series(var_imp).sort_values(ascending=True)
    colors_agg = ["#E24B4A" if v > 0.25 else "#EF9F27" if v > 0.10 else "#3266ad"
                  for v in fi_agg]
    fi_agg.plot(kind="barh", ax=ax, color=colors_agg)
    ax.set_title("Feature Importance (per variabel)
(grouped)", fontweight="bold")
    ax.set_xlabel("Importance")
    for i, v in enumerate(fi_agg):
        ax.text(v + 0.002, i, f"{v*100:.1f}%", va="center", fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} tidak mendukung feature_importances_ langsung.")


## 9. Simpan Model

In [ ]:
import os
os.makedirs("model_output", exist_ok=True)

model_path = f"model_output/best_model_{best_name.lower().replace(' ', '_')}.pkl"
joblib.dump(results[best_name]["pipeline"], model_path)

print(f"✅ Model '{best_name}' disimpan ke: {model_path}")
print(f"\nCara load kembali:")
print(f"  import joblib")
print(f"  pipeline = joblib.load('{model_path}')")


## 10. Prediksi Karyawan Baru

Gunakan sel di bawah untuk memprediksi risiko karyawan baru.  
Ubah nilai sesuai data karyawan yang ingin diperiksa.


In [ ]:
# ── UBAH DATA DI SINI ──────────────────────────────────────────────────────
karyawan_baru = pd.DataFrame([{
    "Usia"                : 50,           # angka (15–70)
    "Jam_Kerja_per_Minggu": 35,           # angka (30–60)
    "Pengalaman_Tahun"    : 35,           # angka (1–60)
    "Pelatihan_K3"        : "Tidak",      # "Ya" atau "Tidak"
    "Shift"               : "Pagi",       # "Pagi", "Siang", "Malam"
    "Jabatan"             : "Operator",   # "Operator", "Supervisor", "Manager"
    "Lokasi"              : "Quality_Control",  # "Gudang", "Produksi", "Quality_Control"
}])
# ───────────────────────────────────────────────────────────────────────────

pipeline  = results[best_name]["pipeline"]
prob      = pipeline.predict_proba(karyawan_baru)[0][1]
label     = pipeline.predict(karyawan_baru)[0]
level     = "TINGGI 🔴" if prob >= 0.6 else "SEDANG 🟡" if prob >= 0.35 else "RENDAH 🟢"

print("=" * 45)
print("  HASIL PREDIKSI RISIKO KARYAWAN")
print("=" * 45)
print(f"  Probabilitas kecelakaan : {prob*100:.1f}%")
print(f"  Prediksi label          : {'Berisiko ⚠️' if label == 1 else 'Aman ✅'}")
print(f"  Level risiko            : {level}")
print("=" * 45)

# Rekomendasi otomatis
recs = []
if karyawan_baru["Lokasi"].iloc[0] == "Quality_Control":
    recs.append("Audit keselamatan menyeluruh di area QC")
if karyawan_baru["Pelatihan_K3"].iloc[0] == "Tidak":
    recs.append("Jadwalkan pelatihan K3 segera")
if karyawan_baru["Shift"].iloc[0] == "Pagi":
    recs.append("Pastikan briefing keselamatan awal shift pagi")
if karyawan_baru["Usia"].iloc[0] >= 45:
    recs.append("Pemeriksaan kesehatan rutin setiap 6 bulan")
if karyawan_baru["Pengalaman_Tahun"].iloc[0] > 30:
    recs.append("Refresh pelatihan prosedur keselamatan terkini")
if karyawan_baru["Jam_Kerja_per_Minggu"].iloc[0] < 40:
    recs.append("Evaluasi konsistensi penugasan dan pemahaman SOP")

if recs:
    print("\n  📋 Rekomendasi tindakan:")
    for r in recs:
        print(f"    → {r}")


In [ ]:
# Prediksi batch — beberapa karyawan sekaligus
batch = pd.DataFrame([
    {"Usia": 50, "Jam_Kerja_per_Minggu": 35, "Pengalaman_Tahun": 35,
     "Pelatihan_K3": "Tidak", "Shift": "Pagi",  "Jabatan": "Operator",   "Lokasi": "Quality_Control"},
    {"Usia": 28, "Jam_Kerja_per_Minggu": 42, "Pengalaman_Tahun":  3,
     "Pelatihan_K3": "Ya",    "Shift": "Siang", "Jabatan": "Operator",   "Lokasi": "Gudang"},
    {"Usia": 40, "Jam_Kerja_per_Minggu": 48, "Pengalaman_Tahun": 15,
     "Pelatihan_K3": "Ya",    "Shift": "Malam", "Jabatan": "Supervisor", "Lokasi": "Produksi"},
    {"Usia": 55, "Jam_Kerja_per_Minggu": 38, "Pengalaman_Tahun": 40,
     "Pelatihan_K3": "Tidak", "Shift": "Pagi",  "Jabatan": "Operator",   "Lokasi": "Quality_Control"},
    {"Usia": 32, "Jam_Kerja_per_Minggu": 44, "Pengalaman_Tahun":  8,
     "Pelatihan_K3": "Ya",    "Shift": "Siang", "Jabatan": "Manager",    "Lokasi": "Gudang"},
])

probs  = pipeline.predict_proba(batch)[:, 1]
labels = pipeline.predict(batch)

batch["Probabilitas (%)"] = (probs * 100).round(1)
batch["Status"]           = ["⚠ BERISIKO" if l == 1 else "✅ AMAN" for l in labels]
batch["Level"]            = ["🔴 TINGGI" if p >= 60 else "🟡 SEDANG" if p >= 35 else "🟢 RENDAH"
                              for p in batch["Probabilitas (%)"]]

display(batch)


## 11. Insight & Kesimpulan

### Temuan Utama

| Aspek | Temuan |
|---|---|
| Tingkat kecelakaan | 28% dari 300 karyawan (84 orang) |
| Lokasi paling berisiko | **Quality Control** (35.2% accident rate) |
| Shift paling berisiko | **Shift Pagi** (31.2%) |
| Jam kerja berbahaya | Karyawan dengan **<40 jam/minggu** memiliki accident rate tertinggi (41%) |
| Faktor individu terpenting | **Usia** dan **Pengalaman Kerja** (kontribusi >25% masing-masing) |
| Dampak Pelatihan K3 | Mengurangi risiko ~4 poin persentase |

### Tentang Performa Model

> **AUC ≈ 0.50–0.53** — model saat ini mendekati kemampuan tebak acak untuk prediksi *per individu*.

Ini bukan karena dataset salah, melainkan karena **distribusi fitur antara kelas 0 dan 1 sangat tumpang tindih** — tidak ada satu "garis pemisah" yang jelas. Kondisi ini umum terjadi pada data keselamatan kerja dengan faktor risiko yang kompleks.

### Rekomendasi Pengembangan

1. **Tambah data** — jenis pekerjaan spesifik, hasil medical checkup, riwayat insiden detail
2. **Gunakan model sebagai alat prioritas kelompok**, bukan prediksi individual
3. **Fokus intervensi** pada: karyawan QC + shift pagi + belum K3 + usia 45–55
4. **Feature engineering** — buat fitur kombinasi (misal: `lokasi × shift`, `usia × jam_kerja`)
